    # Feature Engineering DisasterWatch

    Membentuk fitur waktu, lokasi, dampak, dan agregasi wilayah dari dataset bersih.
    

In [ ]:
    from pathlib import Path
    import pandas as pd
    import numpy as np
    
    ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
    INPUT = ROOT / "data" / "processed" / "bnpb_disaster_clean.csv"
    if not INPUT.exists():
        INPUT = ROOT / "data" / "processed" / "bnpb_disaster_clean.csv"
    OUT = ROOT / "data" / "processed"
    df = pd.read_csv(INPUT, encoding="utf-8-sig", parse_dates=["date"])
    print(df.shape)
    

## 1. Fitur waktu

In [ ]:
    df["year"] = df.date.dt.year
    df["month"] = df.date.dt.month
    df["day"] = df.date.dt.day
    df["day_of_year"] = df.date.dt.dayofyear
    df["week_of_year"] = df.date.dt.isocalendar().week.astype(int)
    df["day_of_week"] = df.date.dt.dayofweek
    df["is_weekend"] = (df.day_of_week >= 5).astype(int)
    df["quarter"] = df.date.dt.quarter
    

## 2. Fitur dampak dan lokasi

In [ ]:
    impact = ["meninggal", "terluka", "hilang", "menderita", "mengungsi", "terendam", "rumah_rusak", "lahan"]
    for col in impact:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df["total_korban"] = df[["meninggal", "terluka", "hilang"]].sum(axis=1)
    df["total_terdampak"] = df[["menderita", "mengungsi"]].sum(axis=1)
    df["total_dampak_fisik"] = df[["rumah_rusak", "terendam", "lahan"]].sum(axis=1)
    df["has_coordinates"] = df[["longitude", "latitude"]].notna().all(axis=1).astype(int)
    

## 3. Fitur frekuensi historis wilayah

In [ ]:
    df = df.sort_values("date")
    df["province_event_count"] = df.groupby("provinsi").cumcount() + 1
    df["kabkot_event_count"] = df.groupby("kabkot").cumcount() + 1
    df["type_event_count"] = df.groupby("kejadian").cumcount() + 1
    

## 4. Agregasi data

In [ ]:
    # Agregasi dipisahkan dari data detail agar bisa dipakai untuk dashboard dan analisis tren.
    agg_metrics = {
        "jumlah_laporan": ("kejadian", "size"),
        "jumlah_kejadian": ("jumlah_kejadian", "sum"),
        "meninggal": ("meninggal", "sum"),
        "terluka": ("terluka", "sum"),
        "hilang": ("hilang", "sum"),
        "menderita": ("menderita", "sum"),
        "mengungsi": ("mengungsi", "sum"),
        "rumah_rusak": ("rumah_rusak", "sum"),
        "terendam": ("terendam", "sum"),
        "lahan": ("lahan", "sum"),
    }

    def aggregate(group_cols, filename):
        result = df.groupby(group_cols, dropna=False).agg(**agg_metrics).reset_index()
        result.to_csv(OUT / filename, index=False, encoding="utf-8-sig")
        print(f"{filename}: {result.shape}")
        return result

    # 1) Tren nasional per tanggal dan per bulan.
    df["year_month"] = df["date"].dt.to_period("M").astype(str)
    daily_agg = aggregate(["date"], "bnpb_aggregate_daily.csv")
    monthly_agg = aggregate(["year", "month", "year_month"], "bnpb_aggregate_monthly.csv")

    # 2) Rekap wilayah dan jenis bencana.
    province_agg = aggregate(["provinsi"], "bnpb_aggregate_province.csv")
    kabkot_agg = aggregate(["provinsi", "kabkot"], "bnpb_aggregate_kabkot.csv")
    type_agg = aggregate(["kejadian"], "bnpb_aggregate_type.csv")

    # 3) Dimensi gabungan untuk heatmap/pivot dashboard.
    province_type_agg = aggregate(["provinsi", "kejadian"], "bnpb_aggregate_province_type.csv")
    province_month_agg = aggregate(["year_month", "provinsi"], "bnpb_aggregate_province_month.csv")
    display(monthly_agg.head(), province_agg.head(), province_type_agg.head())
    

## 5. Encoding kategorikal dan ekspor

In [ ]:
    model_cols = ["date", "year", "month", "day_of_year", "week_of_year", "day_of_week", "is_weekend",
                  "quarter", "provinsi", "kabkot", "kejadian", "longitude", "latitude", "has_coordinates",
                  "total_korban", "total_terdampak", "total_dampak_fisik", "province_event_count",
                  "kabkot_event_count", "type_event_count", "jumlah_kejadian"]
    features = df[[c for c in model_cols if c in df.columns]].copy()
    features.to_csv(OUT / "bnpb_disaster_features.csv", index=False, encoding="utf-8-sig")
    print("Disimpan:", features.shape)
    display(features.head())
    

### Catatan leakage
Fitur agregat harus dihitung hanya dari data masa lalu saat modelling time-series. Jangan memakai kolom dampak jika target yang diprediksi adalah dampak yang sama. File agregasi yang dibuat: `bnpb_aggregate_daily.csv`, `bnpb_aggregate_monthly.csv`, `bnpb_aggregate_province.csv`, `bnpb_aggregate_kabkot.csv`, `bnpb_aggregate_type.csv`, `bnpb_aggregate_province_type.csv`, dan `bnpb_aggregate_province_month.csv`.